In [3]:
import pandas as pd

import numpy as np

from os import makedirs

from utils.utils import nameExtracter

In [4]:
df = pd.read_excel(
    "./../data/raw/B2B/B2B Daily Update sheet.xlsx", sheet_name="ALL DATA"
)

df.columns = [i.strip().title() for i in df.columns]
df = (
    df.dropna(how="all")
    .drop(
        labels=["Sr No", "Total Amount", "Overall"],
        axis=1,
    )
    .dropna(how="all")
)

In [5]:
def formatDate(row: pd.Series) -> pd.Timestamp | float:
    try:
        return pd.to_datetime(row["Date"])
    except:
        return np.nan


df["Date"] = df.apply(formatDate, axis=1).ffill()

In [6]:
df.dropna(subset=df.columns.difference(["Date"]), how="all", inplace=True)

In [7]:
df[["Name", "Address"]] = (
    df[["Name", "Address"]]
    .ffill()
    .apply(
        lambda srs: pd.Series(
            name=srs.name, data=[str(string).title() for string in srs]
        ),
        axis=1,
    )
)

df["Item Name"] = df["Item Name"].apply(lambda name: str(name).title())

In [8]:
df

,Date,Name,Address,Item Name,Qty,Rate,Total Payment,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
1,2025-09-14,Manoj Jaiswal,Sadar,G4 Potato Retail,100.0,23.75,2375.0,NaN,NaN,NaN,NaN
2,2025-09-14,Manoj Jaiswal,Sadar,Red Onion Premium Retail,100.0,18.75,1875.0,NaN,NaN,NaN,NaN
3,2025-09-14,Manoj Jaiswal,Sadar,Nan,NaN,NaN,NaN,,NaN,NaN,NaN
4,2025-09-14,Jagdish Shahu,Besa,Agra Potato Premium Retail,50.0,21.25,1062.5,NaN,NaN,NaN,NaN
5,2025-09-14,Jagdish Shahu,Besa,Red Onion Premium Retail,50.0,18.75,937.5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
564,2025-10-04,Bhiwa Kawle,Laxmi Nagar,G4 Potato Premium Retail,50.0,21.00,1050.0,NaN,NaN,NaN,NaN
565,2025-10-04,Bhiwa Kawle,Laxmi Nagar,Red Onion Premium Retail,50.0,18.95,947.5,NaN,NaN,NaN,NaN
566,2025-10-04,Bhiwa Kawle,Laxmi Nagar,Garlic Regular,5.0,120.00,600.0,NaN,NaN,NaN,NaN
568,2025-10-04,Shailesh Shahu,Khamla,Agra Potato Premium Retail,100.0,21.00,2100.0,NaN,NaN,NaN,NaN


In [9]:
lst = [
    "Red Onion Premium Wholesale",
    "Red Onion Premium Retail",
    "White onion Wholesale",
    "White onion Retail",
    "Agra potato Premium Wholesale",
    "Agra Potato Premium Retail",
    "Agra Potato Retail Big",
    "G4 potato Wholesale",
    "G4 Potato Retail",
    "Ginger Banglore",
    "Garlic Regular",
    "Garlic Bolder",
    "Onion B",
    "Potato B",
    "Ginger B",
    "White Onion B",
    "Potato Small",
]

In [10]:
df = df.dropna(
    subset=[
        "Item Name",
    ],
    how="all",
)

In [11]:
df

,Date,Name,Address,Item Name,Qty,Rate,Total Payment,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
1,2025-09-14,Manoj Jaiswal,Sadar,G4 Potato Retail,100.0,23.75,2375.0,NaN,NaN,NaN,NaN
2,2025-09-14,Manoj Jaiswal,Sadar,Red Onion Premium Retail,100.0,18.75,1875.0,NaN,NaN,NaN,NaN
3,2025-09-14,Manoj Jaiswal,Sadar,Nan,NaN,NaN,NaN,,NaN,NaN,NaN
4,2025-09-14,Jagdish Shahu,Besa,Agra Potato Premium Retail,50.0,21.25,1062.5,NaN,NaN,NaN,NaN
5,2025-09-14,Jagdish Shahu,Besa,Red Onion Premium Retail,50.0,18.75,937.5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
564,2025-10-04,Bhiwa Kawle,Laxmi Nagar,G4 Potato Premium Retail,50.0,21.00,1050.0,NaN,NaN,NaN,NaN
565,2025-10-04,Bhiwa Kawle,Laxmi Nagar,Red Onion Premium Retail,50.0,18.95,947.5,NaN,NaN,NaN,NaN
566,2025-10-04,Bhiwa Kawle,Laxmi Nagar,Garlic Regular,5.0,120.00,600.0,NaN,NaN,NaN,NaN
568,2025-10-04,Shailesh Shahu,Khamla,Agra Potato Premium Retail,100.0,21.00,2100.0,NaN,NaN,NaN,NaN


In [38]:
cust_details = pd.read_csv("./../data/processed/B2B/Customer Details.csv")

In [39]:
cust_details[cust_details["name"].str.contains("Abhi")]

,name,mobile_number,area,city
72,Abhijit Satpute,7.218802e+09,NaN,Nagpur
168,Abhidip Keshwrao Deshpande,9.561813e+09,NaN,Nagpur
252,Abhishek Saroj,8.010134e+09,NaN,Nagpur


In [44]:
ordered = pd.DataFrame(
    columns=["name", "Address"],
    data=[i for i in df.groupby(["Name", "Address"]).groups],
)

In [52]:
res = ordered.merge(
    cust_details,
    left_on="name",
    right_on="name",
    how="outer",
    indicator=True,
)

In [55]:
res = res.drop(["_merge", "area"], axis=1)

In [57]:
res.columns = ["name", "area", "mobile_number", "city"]

In [61]:
res.to_csv("./../data/processed/B2B/to_db.csv", index=False)

In [ ]:
df.to_excel("./../data/processed/B2B/Transaction List.xlsx", index=False)

In [ ]:
grp = df.groupby(["Date"])

grp_names = list(grp.groups.keys())
ne = grp_names[4]
ne, df[df["Date"] == ne].sort_values("Item Name")[
    ["Date", "Item Name", "Rate"]
].groupby("Item Name")["Rate"].unique()

(Timestamp('2025-09-18 00:00:00'),
 Item Name
 Agra Potato Premium Retail     [21.0, 21.25]
 B Potato                              [10.0]
 G4 Big                                [15.0]
 G4 Potato Retail                      [21.0]
 Garlic Bolder                        [120.0]
 Garlic Regular                [100.0, 105.0]
 Ginger Banglore                 [80.0, 85.0]
 Onion B                               [10.0]
 Red Onion Premium Retail      [18.55, 18.75]
 White Onion Retail                    [30.0]
 Name: Rate, dtype: object)